In [ ]:
import numpy as np
import os
from TopsCorr import read_isce
from TopsCorr import utils as ut
from TopsCorr import plot
from TopsCorr import correction as corr
from lsc_lib import gnss_ngl
import h5py
import glob
import datetime as dt

In [ ]:
# input
isce_path = 'p60/tops_ion'
dem_file_isce = os.path.join('dem/demLat_N50_N61_Lon_E154_E164.dem.wgs84.vrt')
output_path = os.path.join(isce_path,'tops_corr_0220')
wavelength = 0.056 # meter
downsampling = True
down_reso = 0.002 # in degree
trop_corr = True # using era5
set_corr = True
coh_thres = 0.5
ref_lonlat = None
ref_gnss = 'PETT'
radius = 400 # meter
to_gnss = True
if to_gnss:
    gnss_path = os.path.join(output_path,'gnss_ngl')
#plot_region = [154.5,163,50.5,61]
plot_region = None

In [3]:
os.makedirs(output_path,exist_ok=True)
data_path = os.path.join(isce_path,'merged/')
time_master = os.path.join(isce_path,'reference/IW1.xml')
time_secondary = os.path.join(isce_path,'secondary/IW1.xml')
tops_para = test = os.path.join(isce_path,'topsApp.xml')
unwrap_file = os.path.join(data_path,'filt_topophase.unw.geo.vrt')
los_file = os.path.join(data_path,'los.rdr.geo.vrt')
coh_file = os.path.join(data_path,'phsig.cor.geo.vrt')

dem_file_wgs84 = os.path.join(output_path,'dem.wgs84.vrt') # this dem is used for tropospheric correction
dem_file_egm = os.path.join(output_path,'dem.egm.vrt') # used for water mask

if downsampling:
    unwrap_small = os.path.join(output_path,'filt_topophase.unw.geo.small.vrt')
    los_small = os.path.join(output_path,'los.rdr.geo.small.vrt')
    coh_small = os.path.join(output_path,'phsig.cor.geo.small.vrt')
    dem_egm_small = os.path.join(output_path,'dem.egm.small.vrt')
    dem_wgs84_small = os.path.join(output_path,'dem.wgs84.small.vrt')

if trop_corr:
    if downsampling:
        era_file = os.path.join(output_path,'era5_small.h5')
    else:
        era_file = os.path.join(data_path,'era5.h5')
if set_corr:
    if downsampling:
        set_file = os.path.join(output_path,'set_small.h5')
    else:
        set_file = os.path.join(data_path,'set.h5')

transform = read_isce.read_data(unwrap_file,band=2)[1]
if not os.path.exists(dem_file_wgs84) or not os.path.exists(dem_file_egm):
    read_isce.gdal_crop(dem_file_isce,dem_file_wgs84,[transform['x0'],transform['y0'],transform['x_end'],transform['y_end']])
    read_isce.gdal_detum_switch(dem_file_wgs84,dem_file_egm,'EPSG:4979','EPSG:4326+5773')   


/Users/kevinwang/miniconda3/envs/tops_corr/lib/python3.14/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [4]:
if downsampling:
    read_isce.gdal_downsample(unwrap_file,unwrap_small,down_reso)
    read_isce.gdal_downsample(los_file,los_small,down_reso)
    read_isce.gdal_downsample(coh_file,coh_small,down_reso)
    read_isce.gdal_downsample(dem_file_egm,dem_egm_small,down_reso)
    read_isce.gdal_downsample(dem_file_wgs84,dem_wgs84_small,down_reso)  
    unwrap_file = unwrap_small
    los_file = los_small
    coh_file = coh_small
    dem_file_egm = dem_egm_small
    dem_file_wgs84 = dem_wgs84_small

In [5]:
# reading data
unwrap, transform = read_isce.read_data(unwrap_file,band=2) 
unwrap = ut.rad2disp(unwrap,wavelength)
dem_egm = read_isce.read_data(dem_file_egm)[0]
dem_wgs84 = read_isce.read_data(dem_file_wgs84)[0]
coh = read_isce.read_data(coh_file)[0]
inc = read_isce.read_data(los_file,band=1)[0]
azi = read_isce.read_data(los_file,band=2)[0] # azimuth angle of measurement rather than along track direction!

t_master = read_isce.get_sensing_time(time_master)
t_sec = read_isce.get_sensing_time(time_secondary)
master_date = t_master.replace(hour=0,minute=0, second=0, microsecond=0)
sec_date = t_sec.replace(hour=0,minute=0, second=0, microsecond=0)
master_yyyymmdd = dt.datetime.strftime(master_date,'%Y%m%d')
sec_yyyymmdd = dt.datetime.strftime(sec_date,'%Y%m%d')

lon_list = transform['x0'] + np.arange(transform['x_size']) * transform['dx']
lat_list = transform['y0'] + np.arange(transform['y_size']) * transform['dy']
lon,lat = np.meshgrid(lon_list,lat_list)
del lon_list, lat_list
region = [transform['x0'],transform['x_end'],transform['y_end'],transform['y0']]

range_looks, azi_looks = read_isce.get_looks(tops_para)

# create masks
water_mask = dem_egm >= 2 # at sea surface there are some weird patterns.
nan_mask = ~np.isnan(unwrap)
coh_mask = coh >= coh_thres
inc_mask = inc >= 20 # not sure why there are very small inc in the farest range.
data_mask = water_mask * nan_mask * coh_mask * inc_mask
del water_mask,nan_mask,coh_mask, dem_egm, inc_mask
inc = np.deg2rad(inc)
azi = np.deg2rad(azi)

In [ ]:
if trop_corr:
    compute_trop = True
    if os.path.exists(era_file):
        with h5py.File(era_file,'r') as r:
            trop_delay = r['trop_delay'][:]
        if trop_delay.shape != unwrap.shape:
            print('Find existing tropospheric correct but with different resolution. Deleting it and computing it again.')
            os.remove(era_file)
        else:
            compute_trop = False
            print('Find existing tropospheric correction. Loading it.')
            
    if compute_trop:
        print('Computing tropospheric correction using PyAPS...')
        trop_delay_m = corr.tropo_corr(t_master,dem_wgs84,np.rad2deg(inc),lon,lat)
        trop_delay_s = corr.tropo_corr(t_sec,dem_wgs84,np.rad2deg(inc),lon,lat)
        trop_delay = trop_delay_m - trop_delay_s
        print(f'Saving tropospheric correction as {os.path.basename(era_file)} in {output_path}')
        with h5py.File(era_file,'w') as f:
            g = f.create_group('geotransform')
            g.create_dataset('x0',data=transform['x0'])
            g.create_dataset('y0',data=transform['y0'])
            g.create_dataset('dx',data=transform['dx'])
            g.create_dataset('dy',data=transform['dy'])
            g.create_dataset('x_size',data=transform['x_size'])
            g.create_dataset('y_size',data=transform['y_size'])
            f.create_dataset('trop_delay',data=trop_delay)
            f.attrs['unit'] = 'meter'
        del trop_delay_m,trop_delay_s, dem_wgs84

Computing tropospheric correction using PyAPS...
INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
{'product_type': ['reanalysis'], 'variable': ['geopotential', 'temperature', 'specific_humidity'], 'year': ['2025'], 'month': ['07'], 'day': ['19'], 'time': ['19:00'], 'pressure_level': ['1', '2', '3', '5', '7', '10', '20', '30', '50', '70', '100', '125', '150', '175', '200', '225', '250', '300', '350', '400', '450', '500', '550', '600', '650', '700', '750', '775', '800', '825', '850', '875', '900', '925', '950', '975', '1000'], 'data_format': 'grib', 'area': [61, 155, 50, 164]}


2026-02-20 14:41:39,435 INFO Request ID is 803e550e-e92a-4f49-bc1c-3de3302ae436
2026-02-20 14:41:39,575 INFO status has been updated to accepted
2026-02-20 14:41:48,563 INFO status has been updated to running
2026-02-20 14:42:01,509 INFO status has been updated to successful


INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
{'product_type': ['reanalysis'], 'variable': ['geopotential', 'temperature', 'specific_humidity'], 'year': ['2025'], 'month': ['07'], 'day': ['19'], 'time': ['20:00'], 'pressure_level': ['1', '2', '3', '5', '7', '10', '20', '30', '50', '70', '100', '125', '150', '175', '200', '225', '250', '300', '350', '400', '450', '500', '550', '600', '650', '700', '750', '775', '800', '825', '850', '875', '900', '925', '950', '975', '1000'], 'data_format': 'grib', 'area': [61, 155, 50, 164]}


2026-02-20 14:42:04,927 INFO Request ID is 2d296973-10a7-4eba-9fd2-cc11eb97d162
2026-02-20 14:42:05,080 INFO status has been updated to accepted
2026-02-20 14:42:19,088 INFO status has been updated to running
2026-02-20 14:42:26,834 INFO status has been updated to successful


INFO: INCIDENCE ANGLE AS AN ARRAY
INFO: AREA COVERAGE IN SNWE: (61.57, 49.80, 154.75, 164.24)
PROGRESS: READING GRIB FILE
INFO: USING PRESSURE LEVELS OF ERA-INT OR ERA-5 DATA
INFO: IMAGE DIMENSIONS: 45 LATITUDES AND 37 LONGITUDES
PROGRESS: INTERPOLATING FROM PRESSURE TO HEIGHT LEVELS
PROGRESS: COMPUTING DELAY FUNCTIONS
PROGRESS: FINE INTERPOLATION OF HEIGHT LEVELS
PROGRESS: CREATE THE BILINEAR INTERPOLATION FUNCTION
PROGRESS: MAPPING THE DELAY
[============================================================]      3s /      0s 
INFO: INCIDENCE ANGLE AS AN ARRAY
INFO: AREA COVERAGE IN SNWE: (61.57, 49.80, 154.75, 164.24)
PROGRESS: READING GRIB FILE
INFO: USING PRESSURE LEVELS OF ERA-INT OR ERA-5 DATA
INFO: IMAGE DIMENSIONS: 45 LATITUDES AND 37 LONGITUDES
PROGRESS: INTERPOLATING FROM PRESSURE TO HEIGHT LEVELS
PROGRESS: COMPUTING DELAY FUNCTIONS
PROGRESS: FINE INTERPOLATION OF HEIGHT LEVELS
PROGRESS: CREATE THE BILINEAR INTERPOLATION FUNCTION
PROGRESS: MAPPING THE DELAY
[=====================

2026-02-20 14:42:39,411 INFO Request ID is c33ba06b-d388-48c2-a585-8a17846d8bf3
2026-02-20 14:42:39,537 INFO status has been updated to accepted
2026-02-20 14:42:53,485 INFO status has been updated to running
2026-02-20 14:43:12,782 INFO status has been updated to successful


INFO: You are using the latest ECMWF platform for downloading datasets:  https://cds.climate.copernicus.eu/api
{'product_type': ['reanalysis'], 'variable': ['geopotential', 'temperature', 'specific_humidity'], 'year': ['2025'], 'month': ['07'], 'day': ['31'], 'time': ['20:00'], 'pressure_level': ['1', '2', '3', '5', '7', '10', '20', '30', '50', '70', '100', '125', '150', '175', '200', '225', '250', '300', '350', '400', '450', '500', '550', '600', '650', '700', '750', '775', '800', '825', '850', '875', '900', '925', '950', '975', '1000'], 'data_format': 'grib', 'area': [61, 155, 50, 164]}


2026-02-20 14:43:15,234 INFO Request ID is 640d9f0e-68cf-47bc-956b-e8b863434614
2026-02-20 14:43:15,376 INFO status has been updated to accepted
2026-02-20 14:43:29,309 INFO status has been updated to running
2026-02-20 14:43:48,589 INFO status has been updated to successful


INFO: INCIDENCE ANGLE AS AN ARRAY
INFO: AREA COVERAGE IN SNWE: (61.57, 49.80, 154.75, 164.24)
PROGRESS: READING GRIB FILE
INFO: USING PRESSURE LEVELS OF ERA-INT OR ERA-5 DATA
INFO: IMAGE DIMENSIONS: 45 LATITUDES AND 37 LONGITUDES
PROGRESS: INTERPOLATING FROM PRESSURE TO HEIGHT LEVELS
PROGRESS: COMPUTING DELAY FUNCTIONS
PROGRESS: FINE INTERPOLATION OF HEIGHT LEVELS
PROGRESS: CREATE THE BILINEAR INTERPOLATION FUNCTION
PROGRESS: MAPPING THE DELAY
[============================================================]      3s /      0s 
INFO: INCIDENCE ANGLE AS AN ARRAY
INFO: AREA COVERAGE IN SNWE: (61.57, 49.80, 154.75, 164.24)
PROGRESS: READING GRIB FILE
INFO: USING PRESSURE LEVELS OF ERA-INT OR ERA-5 DATA
INFO: IMAGE DIMENSIONS: 45 LATITUDES AND 37 LONGITUDES
PROGRESS: INTERPOLATING FROM PRESSURE TO HEIGHT LEVELS
PROGRESS: COMPUTING DELAY FUNCTIONS
PROGRESS: FINE INTERPOLATION OF HEIGHT LEVELS
PROGRESS: CREATE THE BILINEAR INTERPOLATION FUNCTION
PROGRESS: MAPPING THE DELAY
[=====================

In [7]:
if set_corr:
    compute_set = True
    if os.path.exists(set_file):
        
        with h5py.File(set_file,'r') as r:
            set_e = r['set_e'][:]
            set_n = r['set_n'][:]
            set_u = r['set_u'][:]
        if set_e.shape != unwrap.shape:
            print('Find existing SET correct but with different resolution. Deleting it and computing it again.')
            os.remove(set_file)
        else:
            compute_set = False
            print('Find existing SET correction. Loading it.')
    if compute_set:
        print('Computing SET correction using Pysolid...')
        e_m,n_m,u_m = corr.set_corr(t_master,transform['x_size'],transform['y_size'],transform['x0'],transform['y0'],transform['dx'],transform['dy'])
        e_s,n_s,u_s = corr.set_corr(t_sec,transform['x_size'],transform['y_size'],transform['x0'],transform['y0'],transform['dx'],transform['dy'])
        set_e = e_m - e_s
        set_n = n_m - n_s
        set_u = u_m - u_s
        print(f'Saving SET correction as {os.path.basename(set_file)} in {output_path}')
        with h5py.File(set_file,'w') as f:
            g = f.create_group('geotransform')
            g.create_dataset('x0',data=transform['x0'])
            g.create_dataset('y0',data=transform['y0'])
            g.create_dataset('dx',data=transform['dx'])
            g.create_dataset('dy',data=transform['dy'])
            g.create_dataset('x_size',data=transform['x_size'])
            g.create_dataset('y_size',data=transform['y_size'])
            f.create_dataset('set_e',data=set_e)
            f.create_dataset('set_n',data=set_n)
            f.create_dataset('set_u',data=set_u)
            f.attrs['unit'] = 'meter'
        del e_m,n_m,u_m,e_s,n_s,u_s
    set_los = ut.enu2los(set_e,set_n,set_u,inc,azi)

Computing SET correction using Pysolid...
Saving SET correction as set_small.h5 in p60/tops_ion/tops_corr_0220


In [8]:
unwrap = unwrap[data_mask]
lon = lon[data_mask]
lat = lat[data_mask]
inc = inc[data_mask]
azi = azi[data_mask]
coh = coh[data_mask]

In [10]:
if ref_gnss is not None:
    gnss_name,gnss_lat,gnss_lon = gnss_ngl.search_gnss((region[2],region[3],region[0],region[1]), start_date=master_date, end_date=sec_date,output_path=gnss_path)
    gnss_file = [gnss_ngl.download_site(i,gnss_path,'IGS20') for i in gnss_name]
    with open(os.path.join(data_path,'used_gnss_list.txt'),'w') as w:
        for i in gnss_file:
            w.writelines(f"{os.path.abspath(i)}\n")
    gnss_list = glob.glob(os.path.join(gnss_path,'*.tenv3'))
    gnss_list, gnss_lon, gnss_lat, gnss_sta, ind_space_list = gnss_ngl.find_close_sta(gnss_list,lon,lat,master_date,sec_date,radius=radius,time_st_match=True)
    try: 
        ref_ind = gnss_sta.index(ref_gnss)
        ref_lon = gnss_lon[ref_ind]
        ref_lat = gnss_lat[ref_ind]
    except:
        raise Exception(f"The reference gnss {ref_gnss} is not is the SAR frame. Please change the station or specify coordinate.")
else:
    ref_lon = ref_lonlat[0]
    ref_lat = ref_lonlat[1]
unwrap = ut.ref2reference(unwrap,lon,lat,ref_lon,ref_lat,radius=radius)

Don't find ESSO.png
Don't find KAMC.png
Don't find KLCH.png
Don't find KLU2.png
Don't find PETT.png
Don't find PPKK.png


In [11]:
if trop_corr:
    trop_delay = trop_delay[data_mask]
    trop_delay = ut.ref2reference(trop_delay,lon,lat,ref_lon,ref_lat,radius=radius)
    plot.plot_gmt(trop_delay,lon,lat,fig_name='trop_delay',output_path=output_path,ref_lonlat=np.vstack((ref_lon,ref_lat)))
    plot.plot_gmt_subplot(unwrap,unwrap - trop_delay,lon,lat,sub_title=['Ori','Corr trop'],fig_name='trop_corr_check',output_path=output_path)
    unwrap = unwrap - trop_delay

makecpt [WARNING]: Without inc in -T option, -Z has no effect (ignored)


In [12]:
if set_corr:
    set_los = set_los[data_mask]
    set_los = ut.ref2reference(set_los,lon,lat,ref_lon,ref_lat,radius=radius)
    plot.plot_gmt(set_los,lon,lat,fig_name='set',output_path=output_path,ref_lonlat=np.vstack((ref_lon,ref_lat)))
    plot.plot_gmt_subplot(unwrap,unwrap - set_los,lon,lat,sub_title=['Ori','Corr set'],fig_name='set_check',output_path=output_path) # Note the color range depends on the input data, so the color range in this figure is different from one about tropospheric correction. 
    unwrap = unwrap - set_los

makecpt [WARNING]: Without inc in -T option, -Z has no effect (ignored)


In [ ]:
plot.plot_gmt(unwrap,lon,lat,region=plot_region,fig_name=f"{master_yyyymmdd}-{sec_yyyymmdd}",output_path=output_path,ref_lonlat=np.vstack((ref_lon,ref_lat)))
e_unit, n_unit, u_unit = ut.get_unit(inc,azi)
plot.plot_gmt(e_unit,lon,lat,fig_name='e_unit',output_path=output_path)
plot.plot_gmt(n_unit,lon,lat,fig_name='n_unit',output_path=output_path)
plot.plot_gmt(u_unit,lon,lat,fig_name='u_unit',output_path=output_path)

In [16]:
with h5py.File(os.path.join(output_path,'unwrapifg_insarframe.h5'),'w') as f:
    f.create_dataset('lon',data=lon)
    f.create_dataset('lat',data=lat)
    f.create_dataset('disp',data=unwrap)
    f.create_dataset('coh',data=coh)
    f.create_dataset('e_unit',data=e_unit)
    f.create_dataset('n_unit',data=n_unit)
    f.create_dataset('u_unit',data=u_unit)
    f.attrs['unit'] = 'meter'

In [ ]:
if to_gnss:
    if 'gnss_list' not in locals():
        gnss_name,gnss_lat,gnss_lon = gnss_ngl.search_gnss((region[2],region[3],region[0],region[1]), start_date=master_date, end_date=sec_date,output_path=gnss_path)
        gnss_file = [gnss_ngl.download_site(i,gnss_path,'IGS20') for i in gnss_name]
        gnss_list = glob.glob(os.path.join(gnss_path,'*.tenv3'))
        gnss_list, gnss_lon, gnss_lat, gnss_sta, ind_space_list = gnss_ngl.find_close_sta(gnss_list,lon,lat,master_date,sec_date,radius=radius,time_st_match=True)
        if not gnss_list:
            raise('There is no GNSS station in the SAR scene. Skip transformation to GNSS frame')
    
    unwrap_std = np.sqrt(ut.Coh2phaseVar(coh,num_looks=range_looks * azi_looks))
    gnss_insar_los = np.full(shape=(len(gnss_list)), fill_value=np.nan)
    gnss_insar_los_std = gnss_insar_los.copy()
    # create an array to store InSAR data near gnss stations
    insar_gnss_disp = gnss_insar_los.copy()
    insar_gnss_disp_std = gnss_insar_los.copy()
    for i in range(len(gnss_list)):
        inc_avg = inc[ind_space_list[i]].mean()
        azi_avg = azi[ind_space_list[i]].mean()
        gnss_time,e,n,u,e_std,n_std,u_std = gnss_ngl.read_tenv3(gnss_list[i],start_date=master_date,end_date=sec_date)
        e = (e - e[0]) 
        n = (n - n[0])
        u = (u - u[0])
        e_std = e_std 
        n_std = n_std 
        u_std = u_std 
        gnss_los = ut.enu2los(e,n,u,inc_avg,azi_avg)
        gnss_los_std = ut.enu2los_std(e_std,n_std,u_std,inc_avg,azi_avg)
        lia,locb = ut.ismember(gnss_time,[sec_date])
        gnss_insar_los[i] = gnss_los[lia].item()
        gnss_insar_los_std[i] = gnss_los_std[lia].item()
        insar_gnss_disp[i] = unwrap[ind_space_list[i]].mean()
        insar_gnss_disp_std[i] = unwrap_std[ind_space_list[i]].mean()
    G = np.ones((gnss_insar_los_std.shape[0],1))
    num_sta = len(gnss_list)
    p_gnss = np.zeros((num_sta,num_sta))
    p_insar = np.zeros((num_sta,num_sta))
    for i in range(num_sta):
        p_gnss[i,i] = 1/gnss_insar_los_std[i]**2
        p_insar[i,i] = 1/insar_gnss_disp_std[i]**2
    obs = gnss_insar_los - insar_gnss_disp
    P = p_gnss + p_insar
    x = np.linalg.inv(G.T @ P @ G) @ G.T @ P @ obs
    unwrap_gnss = unwrap + x
    plot.plot_gmt(unwrap_gnss,lon,lat,region=plot_region,title=f"{master_yyyymmdd}-{sec_yyyymmdd}",fig_name=f"{master_yyyymmdd}-{sec_yyyymmdd}_GNSS_frame",output_path=output_path)
    with h5py.File(os.path.join(output_path,'unwrapifg_gnssframe.h5'),'w') as f:
        f.create_dataset('lon',data=lon)
        f.create_dataset('lat',data=lat)
        f.create_dataset('disp',data=unwrap_gnss)
        f.create_dataset('coh',data=coh)
        f.create_dataset('e_unit',data=e_unit)
        f.create_dataset('n_unit',data=n_unit)
        f.create_dataset('u_unit',data=u_unit)
        f.attrs['unit'] = 'meter'